In [2]:
# veri seti yükleniyor ve context sayısı 15'ten az olanlar için falback contextler hazırlanıyor
import pandas as pd

df = pd.read_csv("train.csv")

df["context_chunks"] = df.apply(
    lambda row: [
        row["context"][start:end].strip()
        for start, end in zip(
            [0] + list(map(int, row["ctx_split_points"].strip("[]").split(",")))[:-1],
            list(map(int, row["ctx_split_points"].strip("[]").split(",")))
        )
    ], axis=1
)
questions = df["question"].tolist()
answers = df["answer"].tolist()
contexts = df["context_chunks"].tolist()

fallback_ids = ["7665ead7-eef4-4165-a1c1-f6d6897270c7", "b558ed0e-8a0d-46c4-987b-122695f70c4a", "b76a116b-6aaa-4720-85c3-6ea5b4d03557"]
fallback_chunks = []
for fallback_id in fallback_ids:
    fallback_context = df[df["id"] == fallback_id].iloc[0]["context_chunks"]
    fallback_chunks.extend(fallback_context)

print(f"Loaded {len(questions)} questions and contexts.")


Loaded 5999 questions and contexts.


In [3]:
# iki bölüm için de data hazırlanıyor. Pozisyon ve bağlam uzunluğu için

def prepare_contexts(context, correct_chunk, lengths=[1, 5, 10, 15], fallback_chunks=[]):
    prepared_contexts = {}
    for length in lengths:
        if length == 1:
            prepared_contexts[length] = [correct_chunk]
        else:
            selected_context = [None] * length
            middle_index = length // 2
            selected_context[middle_index] = correct_chunk
            available_chunks = [chunk for chunk in context if chunk != correct_chunk]
            left_positions = list(range(middle_index - 1, -1, -1))
            right_positions = list(range(middle_index + 1, length))
            positions = [pos for pair in zip(left_positions, right_positions) for pos in pair]
            for pos in positions:
                if available_chunks:
                    selected_context[pos] = available_chunks.pop(0)
            fallback_index = 0
            for idx, chunk in enumerate(selected_context):
                if chunk is None:
                    if fallback_index < len(fallback_chunks):
                        selected_context[idx] = fallback_chunks[fallback_index]
                        fallback_index += 1
            prepared_contexts[length] = selected_context
    return prepared_contexts

def prepare_positions(context, correct_chunk, total_length=15, fallback_chunks=[]):
    prepared_contexts = {}
    for position in range(total_length):
        new_context = context.copy()
        if len(new_context) < total_length:
            additional_fallbacks = fallback_chunks[:total_length - len(new_context)]
            new_context.extend(additional_fallbacks)
        while correct_chunk in new_context:
            new_context.remove(correct_chunk)
        if position < len(new_context):
            new_context[position] = correct_chunk
        else:
            new_context.append(correct_chunk)
        if len(new_context) < total_length:
            additional_fallbacks = fallback_chunks[:total_length - len(new_context)]
            new_context.extend(additional_fallbacks)
        final_context = new_context[:total_length]
        prepared_contexts[position] = final_context
    return prepared_contexts


In [ ]:
# istenilen ayarlarda ragas sonuçları üretiliyor


import pickle
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, faithfulness, answer_relevancy, context_recall
from ragas.metrics import answer_correctness, faithfulness, answer_relevancy, SemanticSimilarity
from tqdm import tqdm
from datasets import Dataset
import os
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import json

os.environ["OPENAI_API_KEY"] = "REAL_API_KEY"

retrieved_contexts = []
final_question = []

Rtype = "b" # a , b

final_save_path = "gemma/5b/results_b_Gemma2_9b.pkl"

results_save_path = "gemma_b_deneme.json"

if Rtype == "a":
    context_lengths = [1, 5, 10, 15]
    total_questions = 100
else:
    total_questions = 50
    total_length = 15
    
for idx, (question, answer, context) in enumerate(
    tqdm(zip(questions[:total_questions], answers[:total_questions], contexts[:total_questions]), total=total_questions), start=1
):
    correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
    
    if Rtype == "a":
        prepared_contexts = prepare_contexts(context, correct_chunk, lengths=context_lengths, fallback_chunks=fallback_chunks)

        for length, prepared_context in prepared_contexts.items():
            retrieved_contexts.append(prepared_context)
            final_question.append(question)
            
    else:
        prepared_contexts = prepare_positions(context, correct_chunk, total_length=total_length, fallback_chunks=fallback_chunks)

        for position, prepared_context in prepared_contexts.items():
            retrieved_contexts.append(prepared_context)
            final_question.append(question)

print(f"Prepared {len(retrieved_contexts)} sets of retrieved contexts.")
print(len(final_question))



with open(final_save_path, "rb") as f:
    loaded_predictions, loaded_ground_truths = pickle.load(f)
    
print(f"Loaded {len(loaded_predictions)} predictions and ground truths.")
print(len(loaded_ground_truths))

metrics = [answer_correctness, faithfulness, answer_relevancy]

per_sample_results = []

for i in range(5):
    single_data = {
        "question": [final_question[i]],
        "ground_truth": [loaded_ground_truths[i]],
        "answer": [loaded_predictions[i]],
        "retrieved_contexts": [retrieved_contexts[i]]
    }
    dataset = Dataset.from_dict(single_data)

    column_map = {
        "question": "question",
        "ground_truth": "ground_truth",
        "answer": "answer",
        "retrieved_contexts": "retrieved_contexts",
    }

    evaluation_result = evaluate(dataset=dataset, metrics=metrics, column_map=column_map, show_progress=False)

    serialized_result = {
        metric.name: evaluation_result[metric.name]
        for metric in metrics
    }

    per_sample_results.append({
        "index": i,
        "evaluation_result": serialized_result,
        "question": [final_question[i]],
        "ground_truth": [loaded_ground_truths[i]],
        "answer": [loaded_predictions[i]],
        "retrieved_contexts": [retrieved_contexts[i]]
    })
    
    print(f"answer {i + 1} evaluated.")

with open(results_save_path, "w") as json_file:
    json.dump(per_sample_results, json_file, indent=4)

print(f"Per-sample evaluation results saved to {results_save_path}")

In [ ]:
import json

with open('gemma_b_deneme.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data, indent=4, ensure_ascii=False))
